# DeBERTa full walkthrough

Our architecture end to end: **data → encoding → network → training → best checkpoint → temperature calibration → test → hands-on questions**.

```
Question + State + Billing     Question + State + Technical     Question + State + Sales
        │ DeBERTa                      │ DeBERTa (same weights)          │ DeBERTa
        h1                             h2                                h3
        └──────────── self-attention between candidates (no positional encoding) ────────────┘
        h'1                            h'2                               h'3
        │ MLP (shared)                 │ MLP                             │ MLP
        z1                             z2                                z3
        └────────────────── softmax(z / T_task) → P(Billing), P(Technical), P(Sales) ─────────┘
```

* **Encoder**: `architectures/deberta_cross/`. "cross" stands for **cross-encoder**: each candidate is encoded *together* with the context in one sequence (`[CLS] context [SEP] candidate [SEP]`), so their tokens attend to each other. A bi-encoder would embed them separately and compare vectors.
* **Decision head**: `CandidateMLPHead` in `architectures/choice_head.py`.
* **Training**: full fine-tuning (every DeBERTa weight trains, no LoRA) on half of the train split.
* **Four disjoint splits**, split by state text so no state is shared between them:

| split | used for |
|---|---|
| train | gradient updates |
| validation | picking the best checkpoint |
| calibration | fitting one temperature per task type |
| test | the final numbers, evaluated once |

**Reruns are cheap**: if a finished run named `RUN_NAME` exists in `runs/`, training is skipped and that run is loaded. Model outputs are cached next to the checkpoint (`logits_*.pt`), so calibration and test take seconds on a rerun. Set `RETRAIN = True` to train again.

Kernel: select the project's `.venv` (Python 3.10).

In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup

from architectures import get_architecture
from calibration import fit_temperatures, save_temperatures
from dataset import TASK_NAMES, TASK_TYPES, QuestionDataset, load_questions
from inference import answer, autocast, collect_logits, get_device, to_device
from logger import RunLogger, load_checkpoint
from metrics import compute_metrics
from notebook_utils import cached_logits, calibration_curves, free_memory, latest_run, plot_reliability, read_metrics

QUICK_RUN = os.environ.get("NOTEBOOK_QUICK_RUN") == "1"  # tiny sizes, only to check that every cell runs
device = get_device()
torch.manual_seed(0)
pd.options.display.float_format = "{:.4f}".format
print("device:", device, "| quick run:", QUICK_RUN)

## 0. Configuration

The same structure as the YAML files in `configs/`, so the checkpoints saved here also work with `inference.py` and `calibration.py`.

* **`POOLING`** sets how a candidate's sequence becomes `h_i`: the `[CLS]` vector (`cls`), or the mean of the candidate's own tokens (`choice_mean`). `compare_models.ipynb` measures both on the same data; use the winner here.
* `full_finetune: True` trains all ~184M DeBERTa weights with a small learning rate (`lr_backbone` = 3e-5). The freshly initialised head gets 1e-3.
* `batch_size: 4` questions × `grad_accum_steps: 4` = 16 questions per optimizer step. Each question becomes one sequence per candidate.
* `max_state_tokens: 448`, because DeBERTa-v3 was pretrained on 512-token sequences.

⏱ Full fine-tuning takes about **0.25 s per question** on an M5 Pro. `TRAIN_FRACTION = 0.5` (~120K questions) means **about 8–9 hours** of training, and the training cell prints an estimate after 20 steps. `TRAIN_FRACTION = 0.02` takes about 15 minutes.

In [ ]:
RUN_NAME = "deberta-full-walkthrough"   # runs/<RUN_NAME>-<timestamp>/
RETRAIN = False                         # False: reuse the latest finished run with this name if there is one
TRAIN_FRACTION = 0.5                    # share of the train split to train on
POOLING = "choice_mean"                 # "cls" | "choice_mean": pick the winner of compare_models.ipynb

NUM_VALIDATION, NUM_CALIBRATION, NUM_TEST = 1000, 2000, 2000   # questions per evaluation split
EVAL_BATCH_SIZE = 8

CONFIG = {
    "seed": 0,
    "model": {
        "architecture": "deberta_cross",
        "name": "microsoft/deberta-v3-base",
        "num_layers": None,            # all 12 layers
        "pooling": POOLING,
        "full_finetune": True,
    },
    "data": {
        "max_train_questions": None,   # filled in from TRAIN_FRACTION below
        "max_test_questions": NUM_VALIDATION,
        "eval_split": "validation",
        "max_state_tokens": 448,
        "max_choice_tokens": 64,
        "batch_size": 4,
        "num_workers": 0,
    },
    "lora": {"last_k_layers": 0, "r": 8, "alpha": 16, "dropout": 0.0},  # unused with full_finetune
    "head": {"type": "candidate_mlp", "new_dim": 512, "num_layers": 2},
    "optim": {
        "epochs": 1,
        "lr_backbone": 3e-5,
        "lr_head": 1e-3,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "grad_clip": 1.0,
        "grad_accum_steps": 4,
    },
    "logging": {"log_every": 50, "eval_every": 500},  # optimizer steps (16 questions each)
}

if QUICK_RUN:
    RUN_NAME += "-quick"
    TRAIN_FRACTION = 24 / 240_000
    NUM_VALIDATION, NUM_CALIBRATION, NUM_TEST = 16, 40, 24
    CONFIG["data"]["max_test_questions"] = NUM_VALIDATION
    CONFIG["optim"]["grad_accum_steps"] = 2
    CONFIG["logging"].update(log_every=1, eval_every=2)

## 1. The data

[`avbiswas/bev-decision-150K`](https://huggingface.co/datasets/avbiswas/bev-decision-150K): each row is a text **state** with one or more questions about it. Questions come in three types:

| type | answer | candidates we build | the number we use |
|---|---|---|---|
| `choice` | one key of `criteria` | `"key: description"` per key | the chosen key |
| `noul` (yes/no) | true / false | `["No", "Yes"]` | P(yes) |
| `score` | a level index | `"0: level"`, `"1: level"`, … | expected score Σ i·pᵢ |

`load_questions` turns each row into one entry per question and shuffles. Validation and calibration are each about 2% of the dataset's train split (see `dataset.py`).

In [ ]:
d = CONFIG["data"]
train_q = load_questions("train", None, CONFIG["seed"])  # already shuffled
train_q = train_q.select(range(max(1, int(len(train_q) * TRAIN_FRACTION))))
d["max_train_questions"] = len(train_q)  # recorded in the run's config.yaml
val_q = load_questions("validation", NUM_VALIDATION, CONFIG["seed"])
cal_q = load_questions("calibration", NUM_CALIBRATION, CONFIG["seed"])
test_q = load_questions("test", NUM_TEST, CONFIG["seed"])

for name, split in [("train", train_q), ("validation", val_q), ("calibration", cal_q), ("test", test_q)]:
    counts = {TASK_NAMES[t]: split["task_type"].count(t) for t in TASK_NAMES}
    print(f"{name:<12} {len(split):>7} questions  {counts}")

In [ ]:
def show_question(row, state_chars=350):
    label = row["label"]
    print(f"[{TASK_NAMES[row['task_type']]}] {row['instructions']}")
    print("state:", row["state"][:state_chars].replace("\n", " "), "…")
    for i, choice in enumerate(row["choices"]):
        print(f"   {'→' if i == label else ' '} {i}. {choice[:110]}")
    print()

for task in ["choice", "noul", "score"]:
    show_question(next(r for r in val_q if r["task_type"] == TASK_TYPES[task]))

## 2. Encoding: one sequence per candidate

`encode_example` (in `architectures/deberta_cross/tokenization.py`) builds, for **each** candidate:

```
[CLS] <task hint> Question: <instructions> <state> [SEP] <candidate> [SEP]
```

It also records where the candidate's own tokens sit (`choice_spans`), which `choice_mean` pooling uses. Every candidate sees the full context with bidirectional attention, but never the other candidates: comparing them is the decision head's job.

In [ ]:
arch = get_architecture(CONFIG)
tok = arch.tokenization.tokenizer

row = next(r for r in val_q if r["task_type"] == TASK_TYPES["choice"])
ex = arch.encode_example(row["state"], row["instructions"], row["choices"], TASK_NAMES[row["task_type"]],
                         d["max_state_tokens"], d["max_choice_tokens"])
for ids, (start, end) in zip(ex["sequences"], ex["choice_spans"]):
    print(f"{len(ids):>4} tokens | {tok.decode(ids[:12])} … [SEP] | candidate tokens {start}:{end} = {tok.decode(ids[start:end])!r}")

`collate_fn` flattens a batch of B questions into **N = total number of candidates** sequences, and keeps `seq_question_idx` / `seq_choice_idx` so the network can regroup the N embeddings into a `[B, C, H]` tensor for the head. `choice_mask` marks which of the C slots are real (questions have 2 to about 10 candidates).

In [ ]:
make_ds = lambda q: QuestionDataset(q, arch.encode_example, d["max_state_tokens"], d["max_choice_tokens"])
make_loader = lambda ds, **kw: torch.utils.data.DataLoader(ds, collate_fn=arch.collate_fn, **kw)
train_ds, val_ds, cal_ds, test_ds = map(make_ds, [train_q, val_q, cal_q, test_q])

example_batch = arch.collate_fn([train_ds[i] for i in range(4)])
for key, value in example_batch.items():
    print(f"{key:<18} {str(tuple(value.shape)):<12} {value.dtype}")
print("\ncandidates per question:", example_batch["choice_mask"].sum(1).tolist())

## 3. The network

`build_network` loads the pretrained DeBERTa-v3-base weights from the Hugging Face Hub (cached in `~/.cache/huggingface`) and puts our randomly initialised `CandidateMLPHead` on top. The forward pass:

1. DeBERTa encodes the N sequences → hidden states `[N, L, 768]`
2. pooling → one vector per candidate `[N, 768]`, regrouped to `[B, C, 768]`
3. 2 × (self-attention between candidates + MLP), with padded candidates masked out
4. shared scorer MLP 768 → 512 → 1 → logits `[B, C]`

In [ ]:
network = arch.build_network(CONFIG, len(TASK_TYPES)).to(device)

def n_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)

parts = {
    "DeBERTa embeddings": network.backbone.embeddings,
    "DeBERTa encoder (12 layers)": network.backbone.encoder,
    "head: candidate self-attention": network.head.layers,
    "head: scorer MLP": network.head.scorer,
}
for name, module in parts.items():
    print(f"{name:<32} {n_params(module):>12,}")
print(f"{'total':<32} {n_params(network):>12,}  (trainable: {n_params(network, True):,})")

### Layer-by-layer summary

[`torchinfo`](https://github.com/TylerYep/torchinfo) runs one forward pass on the example batch and lists each module with its output shape and parameter count.

In [ ]:
from torchinfo import summary

stats = summary(network, input_data=dict(to_device(example_batch, device)), depth=4, verbose=0,
                col_names=("output_size", "num_params", "trainable"), row_settings=("var_names",))
lines = str(stats).splitlines()
# DeBERTa's 12 encoder layers are identical, so only the first one is printed in full
second = next(i for i, l in enumerate(lines) if "(1)" in l)
last = max(i for i, l in enumerate(lines) if "(11)" in l)
print("\n".join(lines[:second] + ["│    │    │    └─ … layers (1) to (11) are identical …"] + lines[last + 1:]))

### Computation graph (torchview + Graphviz)

[`torchview`](https://github.com/mert-kurttutan/torchview) traces a real forward pass and draws each module with the tensor shapes that flow between them. First the decision head alone: every operation is shown, so you can follow `[B, C, 768]` through the candidate self-attention down to `[B, C]` logits.

In [ ]:
from torchview import draw_graph

H = network.backbone.config.hidden_size
head_inputs = {"choice_embeddings": torch.randn(2, 3, H), "choice_mask": torch.tensor([[True, True, True], [True, True, False]])}
draw_graph(network.head.cpu(), input_data=head_inputs, depth=3, expand_nested=True, graph_name="CandidateMLPHead").visual_graph

The whole network, with DeBERTa collapsed to its two top-level blocks (embeddings, 12-layer encoder). This shows how the N flat sequences become `[N, 768]` candidate vectors, are regrouped to `[B, C, 768]` with `index_put`, and go through the head.

In [ ]:
network.head.to(device)
draw_graph(network, input_data=dict(to_device(example_batch, device)), depth=2, expand_nested=True, device=device,
           graph_name="DebertaCrossNetwork").visual_graph

### Diagram

In [ ]:
from matplotlib.patches import FancyBboxPatch

def box(ax, x, y, w, h, text, color, fontsize=9):
    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc=color, ec="#444", lw=1))
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize)

def arrow(ax, x0, y0, x1, y1):
    ax.annotate("", (x1, y1), (x0, y0), arrowprops=dict(arrowstyle="->", color="#555", lw=1))

def draw_architecture(candidates, cfg):
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.set_xlim(0, 12); ax.set_ylim(0, 10); ax.axis("off")
    xs = [2 + i * 8 / max(len(candidates) - 1, 1) for i in range(len(candidates))]
    pooling = {"cls": "[CLS] vector", "choice_mean": "mean of candidate tokens"}[cfg["model"]["pooling"]]
    for i, (x, c) in enumerate(zip(xs, candidates)):
        box(ax, x, 9.2, 2.6, 0.8, f"[CLS] question + state [SEP]\n{c} [SEP]", "#e8eef7", 8)
        box(ax, x, 7.7, 2.2, 0.8, "DeBERTa-v3-base\n12 layers, shared weights", "#dbe7d5", 8)
        box(ax, x, 6.4, 1.2, 0.6, f"h{i + 1}  (768)", "#fff")
        arrow(ax, x, 8.8, x, 8.1); arrow(ax, x, 7.3, x, 6.7)
        arrow(ax, x, 6.1, x, 5.45)
        box(ax, x, 3.6, 1.2, 0.6, f"h'{i + 1}", "#fff")
        box(ax, x, 2.4, 2.2, 0.8, "MLP 768→512→1\n(shared)", "#f7e3d4", 8)
        arrow(ax, x, 4.75, x, 3.9); arrow(ax, x, 3.3, x, 2.8); arrow(ax, x, 2.0, x, 1.25)
    n = cfg["head"]["num_layers"]
    box(ax, 6, 5.1, 10.6, 0.7, f"{n} × [ self-attention between candidates + MLP ]   (no positional encoding → order-independent)", "#e6dcf0")
    box(ax, 6, 0.9, 10.6, 0.7, "softmax( z / T_task )  →  " + ",  ".join(f"P({c})" for c in candidates), "#f0f0f0", 10)
    ax.set_title(f"deberta_cross + candidate_mlp head   (pooling: {pooling})", fontsize=12)
    plt.show()

draw_architecture(["Billing", "Technical", "Sales"], CONFIG)

## 4. Training: full fine-tuning

**Skipped if a finished run already exists** (see `RETRAIN`). The loop is the same as `train.py`, written out to show each step:

* **Two learning rates**: DeBERTa (pretrained, `lr_backbone`) and the head (random init, `lr_head`), with linear warm-up followed by cosine decay.
* **bf16 autocast** for DeBERTa. The head always runs in fp32.
* **Gradient accumulation**: `grad_accum_steps` micro-batches per optimizer step.
* **Every `eval_every` steps**: evaluate on validation. If validation accuracy improved, save `checkpoints/best`.

Everything is logged to `runs/<RUN_NAME>-<timestamp>/metrics.jsonl`, which section 5 plots. Watch GPU and memory in Activity Monitor → Window → GPU History.

In [ ]:
run_dir = None if RETRAIN else latest_run(RUN_NAME)
if run_dir is not None:
    print(f"Found finished run {run_dir}: skipping training (set RETRAIN = True to train again).")
else:
    o, lg = CONFIG["optim"], CONFIG["logging"]
    logger = RunLogger(CONFIG, RUN_NAME, device)
    run_dir = logger.run_dir
    train_loader = make_loader(train_ds, batch_size=d["batch_size"], shuffle=True)
    val_loader = make_loader(val_ds, batch_size=EVAL_BATCH_SIZE)

    backbone_params = [p for p in network.backbone.parameters() if p.requires_grad]
    head_params = [p for n, p in network.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": o["lr_backbone"]},
        {"params": head_params, "lr": o["lr_head"]},
    ], weight_decay=o["weight_decay"])

    accum = o["grad_accum_steps"]
    steps_per_epoch = len(train_loader) // accum
    total_steps = o["epochs"] * steps_per_epoch
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(o["warmup_ratio"] * total_steps), total_steps)
    print(f"{len(train_ds)} questions → {total_steps} optimizer steps of {d['batch_size'] * accum} questions")

    best_accuracy, best_step, step, window = -1.0, None, 0, []
    start = time.time()
    network.train()
    for epoch in range(o["epochs"]):
        optimizer.zero_grad()
        for micro, batch in enumerate(train_loader, 1):
            if micro > steps_per_epoch * accum:
                break  # drop the incomplete final accumulation group
            batch = to_device(batch, device)
            with autocast(device):
                logits = network(**batch)
            loss = F.cross_entropy(logits, batch["labels"])
            (loss / accum).backward()
            window.append((loss.item(), (logits.argmax(-1) == batch["labels"]).float().mean().item()))
            if micro % accum:
                continue

            torch.nn.utils.clip_grad_norm_(network.parameters(), o["grad_clip"])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step += 1

            if step == 20:
                per_step = (time.time() - start) / step
                print(f"≈ {per_step:.1f} s/step → training ETA {per_step * (total_steps - step) / 3600:.1f} h (+ evaluations)")
            if step % lg["log_every"] == 0:
                train_loss, train_acc = (sum(v) / len(window) for v in zip(*window))
                window = []
                logger.log(step, split="train", loss=train_loss, accuracy=train_acc, lr_backbone=scheduler.get_last_lr()[0])
            if step % lg["eval_every"] == 0 or step == total_steps:
                metrics = compute_metrics(*collect_logits(network, val_loader, device))  # puts the network back in train mode
                logger.log(step, split="validation", **metrics)
                if metrics["accuracy"] > best_accuracy:
                    best_accuracy, best_step = metrics["accuracy"], step
                    logger.save_checkpoint(network, name="best", step=step)

    logger.save_checkpoint(network, name="final", step=step)
    logger.finish(best_validation_accuracy=best_accuracy, best_step=best_step)
    print(f"\nDone in {(time.time() - start) / 60:.1f} min | best validation accuracy {best_accuracy:.3f} at step {best_step}")

## 5. Training curves

Read from the run's `metrics.jsonl`, so this works whether training just ran or was skipped. Train points are averaged over the last `log_every` steps. Validation points are the validation split, and the dashed line marks the step saved as `best`.

In [ ]:
history = read_metrics(run_dir)
tr, va = history["train"], history["validation"]
best_step = json.loads((run_dir / "checkpoints" / "best" / "meta.json").read_text())["step"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot([r["step"] for r in tr], [r["loss"] for r in tr], label="train", alpha=0.8)
axes[0].plot([r["step"] for r in va], [r["loss"] for r in va], "o-", label="validation")
axes[0].set_title("loss (cross-entropy)")
axes[1].plot([r["step"] for r in tr], [r["accuracy"] for r in tr], label="train", alpha=0.8)
axes[1].plot([r["step"] for r in va], [r["accuracy"] for r in va], "o-", label="validation")
axes[1].set_title("accuracy")
for task in TASK_TYPES:
    key = f"accuracy_{task}"
    axes[2].plot([r["step"] for r in va if key in r], [r[key] for r in va if key in r], "o-", label=task)
axes[2].set_title("validation accuracy per task type")
for ax in axes:
    ax.axvline(best_step, ls="--", c="grey", lw=1)
    ax.set_xlabel("optimizer step"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Load the best checkpoint

The checkpoint folder holds `weights.safetensors` (every trainable tensor, so here the full DeBERTa plus the head) and `config.yaml`. `load_checkpoint` rebuilds the network from the config, loads the weights, and applies `calibration.json` if one exists.

In [ ]:
del network
if "optimizer" in globals():
    del optimizer
free_memory(device)

best_dir = run_dir / "checkpoints" / "best"
network, ckpt_config = load_checkpoint(best_dir, device)
print(f"loaded {best_dir} (step {best_step}) | temperatures = {network.temperature}")
print(sorted(p.name for p in best_dir.iterdir()))

## 7. Temperature calibration

The model's outputs are used *as probabilities*: `noul` returns P(yes), and `score` returns Σ i·pᵢ. These numbers only mean something if the model is **calibrated**: of all answers given with 80% confidence, about 80% should be correct. Fine-tuned networks are usually over-confident. Accuracy and AUC, the metrics BEV reports, can't detect this, because they only use the ranking of the probabilities.

**Temperature scaling**: `p = softmax(z / T_task)`, with **one T per task type** (choice, yes/no and score questions have different confidence profiles). Each T minimises the NLL on the **calibration** split, which was used neither for training nor for picking the checkpoint. Dividing logits by T never changes the argmax, so accuracy stays the same.

What we check per task, on the quantity that is actually used:

| task | calibration metric |
|---|---|
| choice | NLL, ECE of the top-choice confidence |
| noul | NLL, **Brier score** and ECE of **P(yes)** |
| score | NLL, **MAE of the expected score**, ECE of the top level |

ECE uses 10 bins and is noisy on a few hundred questions, so read it next to NLL and Brier. Code: `calibration.py` (fitting) and `metrics.py` (metrics).

In [ ]:
cal = cached_logits(network, best_dir, "calibration", make_loader(cal_ds, batch_size=EVAL_BATCH_SIZE), device)
temperatures = fit_temperatures(cal["logits"], cal["labels"], cal["task_types"])
before = compute_metrics(cal["logits"], cal["labels"], cal["task_types"])
after = compute_metrics(cal["logits"], cal["labels"], cal["task_types"], temperatures)

for task, t in temperatures.items():
    print(f"T_{task:<7}= {t:.3f}   ({'softens: was over-confident' if t > 1 else 'sharpens: was under-confident'})")
display(pd.DataFrame({"T = 1": before, "calibrated": after}).T)

save_temperatures(best_dir, temperatures, split="calibration", num_questions=len(cal["labels"]), before=before, after=after)
network.temperature = temperatures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
curves_before = calibration_curves(cal["logits"], cal["labels"], cal["task_types"])
curves_after = calibration_curves(cal["logits"], cal["labels"], cal["task_types"], temperatures)
for ax, task in zip(axes, curves_before):
    p, y, xlabel, ylabel = curves_before[task]
    plot_reliability(ax, p, y, xlabel, ylabel, label="T = 1", show_counts=False, alpha=0.6)
    p, y, *_ = curves_after[task]
    plot_reliability(ax, p, y, xlabel, ylabel, label=f"T = {temperatures[task]:.2f}")
    ax.set_title(f"{task} (calibration split, {len(p)} questions)")
    ax.legend(loc="upper left", fontsize=8)
plt.suptitle("Reliability: points on the diagonal = calibrated, below = over-confident", y=1.02)
plt.tight_layout()
plt.show()

## 8. Test set

Evaluated once, after every choice (checkpoint, temperatures) was made on validation and calibration. The metrics are saved to `test_metrics.json` in the run folder, so they can be compared with other runs.

In [ ]:
test = cached_logits(network, best_dir, "test", make_loader(test_ds, batch_size=EVAL_BATCH_SIZE), device)
test_raw = compute_metrics(test["logits"], test["labels"], test["task_types"])
test_cal = compute_metrics(test["logits"], test["labels"], test["task_types"], temperatures)
display(pd.DataFrame({"T = 1": test_raw, "calibrated": test_cal}).T)
print(f"inference: {test['seconds_per_question'] * 1000:.0f} ms per question")

with open(run_dir / "test_metrics.json", "w") as f:
    json.dump({"checkpoint": str(best_dir), "temperatures": temperatures, "num_questions": len(test["labels"]),
               "uncalibrated": test_raw, "calibrated": test_cal}, f, indent=2)

In [ ]:
curves = calibration_curves(test["logits"], test["labels"], test["task_types"], temperatures)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
names = [t for t in TASK_TYPES if f"accuracy_{t}" in test_cal]
axes[0].bar(names, [test_cal[f"accuracy_{t}"] for t in names], color=["C0", "C1", "C2"][:len(names)])
axes[0].axhline(test_cal["accuracy"], ls="--", c="grey", label=f"overall {test_cal['accuracy']:.3f}")
for i, t in enumerate(names):
    axes[0].text(i, test_cal[f"accuracy_{t}"] + 0.01, f"{test_cal[f'accuracy_{t}']:.3f}", ha="center")
axes[0].set(ylim=(0, 1), title="test accuracy per task type"); axes[0].legend()
for task, (p, y, *_ ) in curves.items():
    plot_reliability(axes[1], p, y, label=task, show_counts=False)
axes[1].set_title("test reliability (calibrated): P(yes) for noul, top confidence otherwise")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 9. Try it yourself

`answer()` (in `inference.py`) takes a question in the dataset's format and returns typed answers using the calibrated temperatures:

* `choice`: `criteria` is a dict `{key: description}` (the description may be empty)
* `score`: `criteria` is a list of level descriptions, lowest first
* `noul`: no criteria, the answer is P(yes)

Change the state, the question or the candidates and re-run the cell.

In [ ]:
def ask(state, instructions, task_type, criteria=None):
    question = {"type": task_type, "instructions": instructions}
    if criteria is not None:
        question["criteria"] = criteria
    result = answer(network, ckpt_config, state, question, device)

    print(f"[{task_type}] {instructions}")
    if task_type == "noul":
        print(f"   P(yes) = {result['noul']:.3f}  {'█' * round(result['noul'] * 30)}")
    else:
        labels = list(criteria) if task_type == "choice" else [f"{i}: {c}" for i, c in enumerate(criteria)]
        probs = result["probabilities"]
        probs = [probs[k] for k in criteria] if isinstance(probs, dict) else probs
        for label, p in zip(labels, probs):
            print(f"   {p:6.3f}  {'█' * round(p * 30):<30}  {label}")
        if task_type == "score":
            print(f"   expected score = {result['score']:.2f}")
    print()
    return result

state = ("Hi, I was charged twice for my March invoice (order #4821). I have contacted you three times already "
         "and nobody answers. If this is not refunded by Friday I will cancel my subscription.")

ask(state, "Which department should handle this ticket?", "choice", {
    "Billing": "payments, invoices, refunds",
    "Technical": "bugs, outages, login problems",
    "Sales": "new purchases, upgrades, pricing questions",
})
ask(state, "Is the customer asking for a refund?", "noul")
ask(state, "How frustrated is the customer?", "score",
    ["calm", "slightly annoyed", "annoyed", "very frustrated", "furious"]);

**Order invariance check**: the head has no positional encoding, so reordering the candidates must give each candidate the same probability.

In [ ]:
criteria = {"Billing": "payments, invoices, refunds", "Technical": "bugs, outages, login problems",
            "Sales": "new purchases, upgrades, pricing questions"}
reordered = {k: criteria[k] for k in ["Sales", "Billing", "Technical"]}
a = ask(state, "Which department should handle this ticket?", "choice", criteria)["probabilities"]
b = ask(state, "Which department should handle this ticket?", "choice", reordered)["probabilities"]
print("max difference:", max(abs(a[k] - b[k]) for k in a))

In [ ]:
# Your own question: edit and run
my_state = "The patient reports chest pain radiating to the left arm, sweating and shortness of breath for 20 minutes."
ask(my_state, "What is the most appropriate next step?", "choice", {
    "Call emergency services": "",
    "Book a routine appointment": "",
    "Rest and drink water": "",
});

## Next steps

* **Compare architectures**: `compare_models.ipynb` puts this model next to Qwen/BEV and the other pooling, all trained on the same data.
* **Ablations**: set `head.num_layers = 0` to score each candidate without any candidate interaction, or `head.type = "bilinear_answer"` to use the BEV head.
* **More data**: `TRAIN_FRACTION = 1.0` uses all ~240K questions. That's best on a cloud GPU (see `cloud_startup_script.sh`).